In [1]:
import pandas as pd
import numpy as np

# preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# model validation
from sklearn.model_selection import StratifiedKFold

# evaluation
from sklearn.metrics import classification_report, accuracy_score, f1_score

# model
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

In [2]:
# Load processed datasets from the previous notebook
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (69979, 175)
Validation shape: (14995, 175)
Test shape: (14996, 175)


In [3]:
# Define target column
target_column = "disease_encoded"

# Split features and target
X_train = train_df.drop(columns=[target_column])
y_train = train_df[target_column]

X_val = val_df.drop(columns=[target_column])
y_val = val_df[target_column]

X_test = test_df.drop(columns=[target_column])
y_test = test_df[target_column]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (69979, 174)
y_train shape: (69979,)
X_val shape: (14995, 174)
y_val shape: (14995,)
X_test shape: (14996, 174)
y_test shape: (14996,)


In [ ]:
def evaluate_model(model, X_train, y_train, X_val, y_val, model_name,sample_weight=None):

     # Train model
    if sample_weight is not None:
        model.fit(X_train, y_train, sample_weight=sample_weight)
    else:
        model.fit(X_train, y_train)

    
    # Predict on validation set
    y_val_pred = model.predict(X_val)
    
    # Compute metrics
    accuracy = accuracy_score(y_val, y_val_pred)
    f1_weighted = f1_score(y_val, y_val_pred, average="weighted")
    f1_macro = f1_score(y_val, y_val_pred, average="macro")
    
    # Store results
    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Weighted F1": f1_weighted,
        "Macro F1": f1_macro
    }
    
    return results, y_val_pred

# Initialize results storage (run once at top of notebook)
results_list = []

In [ ]:
# Compute class weights
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

# Convert to dictionary
class_weights = dict(zip(classes, weights))

# Create sample weights for each training example
sample_weights = np.array([class_weights[y] for y in y_train])


print("Min weight:", sample_weights.min())
print("Max weight:", sample_weights.max())

In [ ]:
num_class = y_train.nunique()
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=num_class,
    eval_metric="mlogloss",
    random_state=42,
    tree_method="hist"
    "n_jobs": -1
)

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 1.5, 2]
}

In [ ]:
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=ps,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train_val, y_train_val, sample_weight=sample_weights_train_val)

print("Best validation Macro F1:", search.best_score_)
print("Best params:", search.best_params_)




In [ ]:
def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_Macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall_Macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1_Macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Precision_Weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall_Weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1_Weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

In [ ]:
best_model_ps = search.best_estimator_
y_pred = best_model_ps.predict(X_test)


In [ ]:
y_pred_ps_val = best_model_ps.predict(X_val)

ps_val_result = evaluate_model(
    "PS Model (Validation)",
    y_val_enc,
    y_pred_ps_val
)

print(ps_val_result)

In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 1.5, 2]
}

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

f1_macro_scorer = make_scorer(f1_score, average="macro")

In [ ]:
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring=f1_macro_scorer,
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    refit=True
)

In [ ]:
search.fit(
    X_train,
    y_train_enc,
    sample_weight=sample_weights
)

In [ ]:
print("Best CV Macro F1:", search.best_score_)
print("Best Parameters:")
print(search.best_params_)

In [ ]:
best_model = search.best_estimator_

In [ ]:
y_pred_val = best_model.predict(X_val)

val_accuracy = accuracy_score(y_val_enc, y_pred_val)
val_precision_macro = precision_score(y_val_enc, y_pred_val, average="macro", zero_division=0)
val_recall_macro = recall_score(y_val_enc, y_pred_val, average="macro", zero_division=0)
val_f1_macro = f1_score(y_val_enc, y_pred_val, average="macro", zero_division=0)

val_precision_weighted = precision_score(y_val_enc, y_pred_val, average="weighted", zero_division=0)
val_recall_weighted = recall_score(y_val_enc, y_pred_val, average="weighted", zero_division=0)
val_f1_weighted = f1_score(y_val_enc, y_pred_val, average="weighted", zero_division=0)

print("Validation Accuracy:", val_accuracy)
print("Validation Precision Macro:", val_precision_macro)
print("Validation Recall Macro:", val_recall_macro)
print("Validation F1 Macro:", val_f1_macro)
print("Validation Precision Weighted:", val_precision_weighted)
print("Validation Recall Weighted:", val_recall_weighted)
print("Validation F1 Weighted:", val_f1_weighted)

In [2]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    make_scorer
)
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, PredefinedSplit
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

In [3]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

In [4]:
X_train = train_df.drop(columns=["disease_encoded"])
y_train = train_df["disease_encoded"]

X_val = val_df.drop(columns=["disease_encoded"])
y_val = val_df["disease_encoded"]

X_test = test_df.drop(columns=["disease_encoded"])
y_test = test_df["disease_encoded"]

In [5]:
y_train_enc = y_train
y_val_enc = y_val
y_test_enc = y_test

In [6]:
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_enc
)

print("Min weight:", sample_weights.min())
print("Max weight:", sample_weights.max())

Min weight: 0.12687491274732168
Max weight: 236.41554054054055


In [8]:
test_fold = np.concatenate([
    -1 * np.ones(len(X_train), dtype=int),   # training fold
    np.zeros(len(X_val), dtype=int)          # validation fold
])

ps = PredefinedSplit(test_fold=test_fold)

In [9]:
num_class = y_train.nunique()
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

In [ ]:
param_dist = {
     "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 1.5, 2]
    
}

In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [ ]:
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    refit=True
)

search.fit(
    X_train,
    y_train_enc,
    sample_weight=sample_weights
)

print("Best validation Macro F1:", search.best_score_)
print("Best params:", search.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


[CV] END colsample_bytree=0.9, gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100, reg_alpha=0.01, reg_lambda=2, subsample=0.9; total time= 2.6min
[CV] END colsample_bytree=0.9, gamma=0.3, learning_rate=0.2, max_depth=5, min_child_weight=5, n_estimators=200, reg_alpha=0.1, reg_lambda=2, subsample=0.7; total time= 5.2min
[CV] END colsample_bytree=0.9, gamma=0.3, learning_rate=0.2, max_depth=5, min_child_weight=5, n_estimators=200, reg_alpha=0.1, reg_lambda=2, subsample=0.7; total time= 5.2min
[CV] END colsample_bytree=0.9, gamma=0.3, learning_rate=0.2, max_depth=5, min_child_weight=5, n_estimators=200, reg_alpha=0.1, reg_lambda=2, subsample=0.7; total time= 5.2min
[CV] END colsample_bytree=0.9, gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100, reg_alpha=0.01, reg_lambda=2, subsample=0.9; total time= 2.6min
[CV] END colsample_bytree=0.9, gamma=0.3, learning_rate=0.2, max_depth=5, min_child_weight=5, n_estimators=200, reg_alpha=0.1

In [27]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

results = {
    "Accuracy": accuracy_score(y_test_enc, y_pred),
    
    
    "F1_Macro": f1_score(y_test_enc, y_pred, average="macro", zero_division=0),
    
    "F1_Weighted": f1_score(y_test_enc, y_pred, average="weighted", zero_division=0),
}

pd.DataFrame([results])

AttributeError: 'RandomizedSearchCV' object has no attribute 'best_estimator_'

In [21]:
param_dist = {
    "n_estimators": [200, 400, 600, 800, 1000],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7, 10],
    "gamma": [0, 0.1, 0.3, 0.5, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 0.5, 1.0],
    "reg_lambda": [1, 2, 5, 10]
}

In [22]:
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=num_class,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1,
    tree_method="hist"
)

In [23]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


In [24]:
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    refit=True
)

search.fit(
    X_train,
    y_train_enc,
    sample_weight=sample_weights
)

print("Best validation Macro F1:", search.best_score_)
print("Best params:", search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


[CV] END colsample_bytree=0.8, gamma=1.0, learning_rate=0.05, max_depth=4, min_child_weight=10, n_estimators=400, reg_alpha=1.0, reg_lambda=1, subsample=0.6; total time= 8.9min
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.01, max_depth=5, min_child_weight=7, n_estimators=400, reg_alpha=0.5, reg_lambda=10, subsample=0.9; total time= 9.3min
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.01, max_depth=5, min_child_weight=7, n_estimators=400, reg_alpha=0.5, reg_lambda=10, subsample=0.9; total time= 9.4min
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.01, max_depth=5, min_child_weight=7, n_estimators=400, reg_alpha=0.5, reg_lambda=10, subsample=0.9; total time= 9.4min
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.03, max_depth=5, min_child_weight=5, n_estimators=400, reg_alpha=0.1, reg_lambda=10, subsample=0.6; total time= 9.6min
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.03, max_depth=5, min_child_weight=5, n_estimators=400, reg_alph

KeyboardInterrupt: 

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

results = {
    "Accuracy": accuracy_score(y_test_enc, y_pred),
    
    
    "F1_Macro": f1_score(y_test_enc, y_pred, average="macro", zero_division=0),
    
    "F1_Weighted": f1_score(y_test_enc, y_pred, average="weighted", zero_division=0),
}

pd.DataFrame([results])